[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_sdf_replication/MFM_ch13_sdf_replication.ipynb)

# MFM_ch13_sdf_replication

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Case study Chen, Pelger and Zhu (2024), Deep Learning in Asset Pricing: monthly out-of-sample Sharpe ratios of the SDF portfolios on 1992-2016 (GAN, elastic net, feedforward network, linear SDF, FF5 and FF3 tangency; Table I and Table A.VI) next to a replication of the FF3 and FF5 tangency portfolios with weights fixed on 1967-1986 (Kenneth French factors, 95% CI from the Lo (2002) standard error), and the cumulative excess returns of these fixed-weight portfolios from 1992 to the latest month, after the end of the paper sample.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip -q install shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Style and data

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Helper functions

In [ ]:
FRENCH_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'

FF_FILES = {'ff3': 'F-F_Research_Data_Factors_CSV.zip', 'ff5': 'F-F_Research_Data_5_Factors_2x3_CSV.zip'}

CPZ_SR = {'GAN': 0.75, 'EN': 0.5, 'FFN': 0.44, 'LS': 0.42, 'FF5': 0.22, 'FF3': 0.19}

CPZ_TRAIN = ('1967-01', '1986-12')

CPZ_TEST = ('1992-01', '2016-12')

FF_COLS = {'ff3': ['Mkt-RF', 'SMB', 'HML'], 'ff5': ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']}

def french_factors(name='ff5'):
    """Factorii lunari FF3 / FF5 (in zecimal, index = sfarsitul lunii) din Kenneth French Data Library."""
    import io
    import zipfile
    import urllib.request
    req = urllib.request.Request(FRENCH_URL + FF_FILES[name], headers={'User-Agent': 'Mozilla/5.0'})
    zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(req, timeout=120).read()))
    lines = zf.read(zf.namelist()[0]).decode('latin-1').splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip() for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        parts = [x.strip() for x in l.split(',')]
        if len(parts[0]) != 6 or not parts[0].isdigit():      # sfarsitul tabelului lunar
            break
        rows.append([parts[0]] + [float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    df['date'] = pd.to_datetime(df['date'], format='%Y%m') + pd.offsets.MonthEnd(0)
    return df.set_index('date') / 100


def tangency_portfolio(factors, train=('1967-01', '1986-12')):
    """Portofoliul tangent cu ponderi fixe w = Sigma^{-1} mu estimate pe perioada de antrenare
    (normalizate la sum|w| = 1); intoarce ponderile si randamentul lunar pe tot esantionul."""
    X = factors.loc[train[0]:train[1]]
    w = np.linalg.solve(X.cov().values, X.mean().values)
    w = pd.Series(w / np.abs(w).sum(), index=factors.columns)
    return w, factors @ w


def monthly_sr(r):
    """Sharpe lunar si eroarea standard Lo (2002) sub randamente i.i.d.: sqrt((1 + SR^2/2) / T)."""
    r = r.dropna()
    sr = r.mean() / r.std(ddof=1)
    return sr, np.sqrt((1 + sr ** 2 / 2) / len(r)), len(r)


def cpz_tangency_returns():
    """Randamentele lunare ale portofoliilor tangente FF3 / FF5 cu ponderi fixate pe 1967-1986, si Mkt-RF."""
    out = {}
    for k in ('ff3', 'ff5'):
        _, out[k.upper()] = tangency_portfolio(french_factors(k)[FF_COLS[k]], CPZ_TRAIN)
    out['Market'] = french_factors('ff3')['Mkt-RF']
    return pd.DataFrame(out).dropna()

## Charts

In [ ]:
def fig_cpz_sharpe(rets):
    """Bare orizontale: SR lunar out-of-sample din lucrare vs. replicarea FF3 / FF5 pe biblioteca French."""
    test = rets.loc[CPZ_TEST[0]:CPZ_TEST[1]]
    ours = {k: monthly_sr(test[k]) for k in ('FF5', 'FF3')}
    labels = list(CPZ_SR)
    y = np.arange(len(labels))[::-1]
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    ax.barh(y + 0.18, [CPZ_SR[k] for k in labels], height=0.34, color=MainBlue,
            label='Chen, Pelger & Zhu (2024), Table I and Table A.VI')
    for k in ('FF5', 'FF3'):
        i = labels.index(k)
        sr, se, T = ours[k]
        ax.barh(y[i] - 0.18, sr, height=0.34, color=IDAred, xerr=1.96 * se,
                error_kw=dict(ecolor='black', lw=0.8, capsize=2.5),
                label=f'Replication: Kenneth French factors, weights 1967-1986, test 1992-2016 (95% CI)'
                if k == 'FF5' else None)
        ax.text(sr + 1.96 * se + 0.015, y[i] - 0.18, f'{sr:.2f}', va='center', fontsize=8, color=IDAred)
    for i, k in enumerate(labels):
        ax.text(CPZ_SR[k] + 0.015 if k not in ('FF5', 'FF3') else CPZ_SR[k] + 0.015, y[i] + 0.18,
                f'{CPZ_SR[k]:.2f}', va='center', fontsize=8, color=MainBlue)
    ax.set_yticks(y)
    ax.set_yticklabels(['GAN (deep SDF)', 'EN (linear SDF, elastic net)', 'FFN (return forecast)',
                        'LS (linear SDF)', 'FF5 tangency', 'FF3 tangency'])
    ax.set_xlim(0, 0.85)
    ax.set_xlabel('Monthly out-of-sample Sharpe ratio, 1992-2016')
    ax.set_title('Deep SDF vs. benchmarks: monthly Sharpe ratio of the SDF portfolio, test 1992-2016',
                 fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=1, y=-0.2)
    plt.tight_layout()
    save_fig('ch13_cpz_sharpe')
    return {k: tuple(round(v, 3) for v in ours[k][:2]) + (ours[k][2],) for k in ours}

In [ ]:
def fig_cpz_tangency_oos(rets):
    """Randament excedentar cumulat (suma) al portofoliilor tangente fixe, scalate la volatilitatea pietei,
    in perioada de test a lucrarii (1992-2016) si dupa ea (2017 - ultima luna)."""
    oos = rets.loc[CPZ_TEST[0]:]
    vol = oos.loc[:CPZ_TEST[1]].std()
    scaled = oos * (vol['Market'] / vol)
    colors = {'FF5': MainBlue, 'FF3': Forest, 'Market': Amber}
    names = {'FF5': 'FF5 tangency', 'FF3': 'FF3 tangency', 'Market': 'Market (Mkt-RF)'}
    table = {}
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    for k in ('FF5', 'FF3', 'Market'):
        a = monthly_sr(oos.loc[:CPZ_TEST[1], k])[0]
        b = monthly_sr(oos.loc['2017-01':, k])[0]
        table[k] = (round(a, 3), round(b, 3))
        ax.plot(scaled.index, scaled[k].cumsum(), color=colors[k], lw=1.0,
                label=f'{names[k]} (monthly SR {a:.2f} in 1992-2016, {b:.2f} in 2017-{oos.index[-1].year})')
    ax.axvline(pd.Timestamp('2016-12-31'), color=Gray, lw=0.7, ls='--')
    ax.axhline(0, color=Gray, lw=0.5)
    ax.text(pd.Timestamp('2016-06-30'), ax.get_ylim()[1] * 0.95, 'end of the paper\'s test period ',
            ha='right', va='top', fontsize=8, color='black')
    ax.set_ylabel('Cumulative excess return\n(scaled to market volatility)')
    ax.set_title(f'Tangency portfolios with weights fixed on 1967-1986: {oos.index[0]:%b %Y} - '
                 f'{oos.index[-1]:%b %Y}', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=1, y=-0.13)
    plt.tight_layout()
    save_fig('ch13_cpz_tangency_oos')
    return table, len(oos.loc['2017-01':]), oos.index[-1]

## Run

In [ ]:
rets = cpz_tangency_returns()
print('Replication (SR, SE, T):', fig_cpz_sharpe(rets))
print('SR 1992-2016 / 2017-latest:', fig_cpz_tangency_oos(rets))

![ch13_cpz_sharpe](./ch13_cpz_sharpe.png)

![ch13_cpz_tangency_oos](./ch13_cpz_tangency_oos.png)

Output: `ch13_cpz_sharpe.pdf`, `ch13_cpz_tangency_oos.pdf`